# ClauseWise — Training Notebook

This notebook covers data preprocessing, model fine-tuning, evaluation, and pushing
trained weights to HuggingFace Hub.

**Inference logic lives in `src/`.** See `app.py` for the FastAPI server.

| Model | HuggingFace Hub ID | Purpose |
|---|---|---|
| LegalBERT | `ClauseWise/legalbert-clause-classifier` | 33-label multi-label clause classification |
| FLAN-T5 + LoRA | `ClauseWise/flan-t5-cuad-clause-extractor-lora` | Clause-level information extraction |

**Reproducing the data:**
Place `master_clauses.csv` (downloaded from the [Atticus Project](https://www.atticusprojectai.org/cuad), specifically their [CUAD HuggingFace Dataset](https://huggingface.co/datasets/theatticusproject/cuad/tree/main) in the CUAD_v1/ folder)
into the `data/` directory, then run cells top-to-bottom.
The `data/` directory is gitignored — it is not committed to the repo.

# **Setup**

In [ ]:
# Setup stuff
!git clone https://github.com/justin73939/ClauseWise.git

In [ ]:
%cd /content/ClauseWise
!pip install -r requirements.txt
!pip install -q --upgrade transformers datasets accelerate sentencepiece peft huggingface_hub

In [ ]:
# Imports
import os
from collections import defaultdict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from transformers import T5ForConditionalGeneration, default_data_collator
from transformers import AutoModelForSeq2SeqLM, AutoModelForCausalLM
from google.colab import files
from tqdm import tqdm
import random
import pandas as pd
import re
from datasets import Dataset
import numpy as np
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import TensorDataset, DataLoader
from torch.nn import BCEWithLogitsLoss
import torch.nn.functional as F
from torch.optim import AdamW
from sklearn.metrics import f1_score
from peft import LoraConfig, get_peft_model
import evaluate
from clause_segmenter import ContractSegmenter, load_contract_text

# **Data Preprocessing**

In [ ]:
# Upload your local CSV (downloaded from Atticus site)
uploaded = files.upload()

In [ ]:
# Load the master CSV you uploaded
df = pd.read_csv("data/master_clauses.csv")
print("Shape before flattening:", df.shape)

# Flatten: one clause per row
long_rows_classif = []
long_rows_textgen = []
for _, row in df.iterrows():
  filename = row["Filename"]
  for col in df.columns:
    if col.endswith("-Answer"):  # find every label column
        category = col.replace("-Answer", "").strip()
        text_col = category
        if text_col not in df.columns:
            continue
        text = row[text_col]
        answer = row[col]

        # skip empty entries
        if pd.isna(text) or text in ([], "", None): continue

        # clean stringified lists
        if isinstance(text, str):
            text = text.strip("[]").replace("'", "").strip()
        if isinstance(answer, str):
            answer = answer.strip("[]").replace("'", "").strip()

        # Separate the 8 columns that do not have yes/no answers for the text generator
        if category in ["Document Name", "Parties", "Agreement Date", "Effective Date", "Expiration Date", "Renewal Term",
                        "Notice Period To Terminate Renewal", "Governing Law"]:
            long_rows_textgen.append({
              "document_name": filename,
              "category": category,
              "text": text,
              "answer": answer
          })
            continue

        long_rows_classif.append({
            "document_name": filename,
            "category": category,
            "text": text,
            "answer": answer
        })

flat_df_classif = pd.DataFrame(long_rows_classif)
flat_df_textgen = pd.DataFrame(long_rows_textgen)
print("Flattened shape:", flat_df_classif.shape)
print("Flattened shape:", flat_df_textgen.shape)

# Clean up text artifacts
def clean_clause(t):
    if pd.isna(t): return ""
    t = re.sub(r"<omitted>", " ", t)
    t = re.sub(r"\[\*+\]", " ", t)
    t = re.sub(r"_+", " ", t)
    t = re.sub(r"\s+", " ", t)
    return t.strip()

flat_df_classif["text"] = flat_df_classif["text"].apply(clean_clause)
flat_df_classif["answer"] = flat_df_classif["answer"].apply(clean_clause)

flat_df_textgen["text"] = flat_df_textgen["text"].apply(clean_clause)
flat_df_textgen["answer"] = flat_df_textgen["answer"].apply(clean_clause)

# Drop empties and tiny fragments
flat_df_classif = flat_df_classif.dropna(subset=["text", "category"])
flat_df_classif = flat_df_classif[flat_df_classif["text"].str.len() > 5]
flat_df_classif = flat_df_classif.drop_duplicates(subset=["text", "category"])

flat_df_textgen = flat_df_textgen.dropna(subset=["text", "category"])
flat_df_textgen = flat_df_textgen[flat_df_textgen["text"].str.len() > 5]
flat_df_textgen = flat_df_textgen.drop_duplicates(subset=["text", "category"])

# Save cleaned file
flat_df_classif.to_csv("data/cuad_flattened_classification.csv", index=False)
print("Saved → data/cuad_flattened_classification.csv")
flat_df_textgen.to_csv("data/cuad_flattened_text_generation.csv", index=False)
print("Saved → data/cuad_flattened_text_generation.csv")

#print(flat_df_classif.head(50))
print(flat_df_textgen.head(50))

In [ ]:
df_class = pd.read_csv("data/cuad_flattened_classification.csv")
print(df_class.shape)
print(df_class.columns)
print(df_class.category.value_counts().head())

categories = df_class["category"].unique()
print(f"\nNumber of unique categories: {len(categories)}")
print(categories)

In [ ]:
# Normalize answers
df_class["label"] = df_class["answer"].str.lower().map({"yes" : 1.0, "no" : 0.0}) # new column added
print(df_class.columns, "\n")
print(set(["label", "text"]).issubset(df_class.columns)) # Check if for every label there is a text

#print(df_class["category"])
print(len(df_class["text"]))
print(len(df_class["text"].unique()))

# For a multilabel model, change how the classification data is set up
# Format: {text : [array of labels encoded directly to categories in order]}

categories = df_class['category'].unique()
print(categories)
category_to_index = {c:i for i, c in enumerate(categories)}

# key=text, value=one-hot array of categories
num_categories = len(categories)
text_to_onehot = {}
for _, row in df_class.iterrows():
    text = row['text']
    category = row['category']
    label = row['label']  # 0 or 1

    if text not in text_to_onehot:
        text_to_onehot[text] = np.zeros(num_categories, dtype=float)

    # Update the corresponding category index with the label
    text_to_onehot[text][category_to_index[category]] = label

# Check to see if it is the same number of unique texts (should be)
print(len(text_to_onehot))


# Convert to final DataFrame
final_df_class = pd.DataFrame({
    'text': list(text_to_onehot.keys()),
    'labels': list(text_to_onehot.values())
})

print(final_df_class.head())

# **Preparing data for the model**

In [ ]:
# Split into train, validation, and test sets
texts = final_df_class["text"].tolist()

# Random state's number doesn't matter, it's purely a seed to reliably make operations reproducible
# Random state shuffles the data in a way depending on the number, the shuffling itself
# does not matter, so the number doesn't matter so long as it is consistent.
# 80% train, 10% validation, 10% test split
train_texts, val_test_texts = train_test_split(texts, test_size=0.2, random_state=0)
val_texts, test_texts = train_test_split(val_test_texts, test_size=0.5, random_state=0)

train_df = final_df_class[final_df_class["text"].isin(train_texts)].reset_index(drop=True)
val_df = final_df_class[final_df_class["text"].isin(val_texts)].reset_index(drop=True)
test_df = final_df_class[final_df_class["text"].isin(test_texts)].reset_index(drop=True)

print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))

In [ ]:
# Tokenize inputs
clausewise_class = "ClauseWise/legalbert-clause-classifier"
tokenizer = AutoTokenizer.from_pretrained(clausewise_class)

def tokenize_texts(texts, max_length=512):
  return tokenizer(
      texts.to_list(),    # Pandas series is converted to a list
      padding="max_length", # Ensure all sequences are not shorter than max_length
      truncation=True,      # Ensure all sequences are not longer than max_length
      max_length=max_length,
      return_tensors="pt" # Returns pytorch tensors for the model
  )

# Convert texts and labels to tensors
train_encodings = tokenize_texts(train_df["text"])
val_encodings = tokenize_texts(val_df["text"])
test_encodings = tokenize_texts(test_df["text"])

# np.array used to speed up runtime when converting to tensor
train_labels = torch.tensor(np.array(list(train_df["labels"].values)), dtype=torch.float)
val_labels = torch.tensor(np.array(list(val_df["labels"].values)), dtype=torch.float)
test_labels = torch.tensor(np.array(list(test_df["labels"].values)), dtype=torch.float)

In [ ]:
# Create PyTorch datasets so the model can use them for training

# input_ids: Numerical representation of words after tokenization
# attention_mask: Tells the model which tokens are real and which are padding (if padding, ignore (basically))
train_dataset = TensorDataset(train_encodings["input_ids"], train_encodings["attention_mask"], train_labels)
val_dataset = TensorDataset(val_encodings["input_ids"], val_encodings["attention_mask"], val_labels)
test_dataset = TensorDataset(test_encodings["input_ids"], test_encodings["attention_mask"], test_labels)

# **Classification (LegalBERT) Model Fine-Tuning Loop**

In [ ]:
# Create DataLoaders
batch_size = 8

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

loss_func = BCEWithLogitsLoss()

# Load model
model = AutoModelForSequenceClassification.from_pretrained(
    clausewise_class,
    num_labels=train_labels.shape[1],
    problem_type="multi_label_classification"
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

# Scheduler
epochs = 3
total_steps = len(train_loader) * epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

for epoch in range(epochs):
  print(f"\n=== Epoch {epoch + 1}/{epochs} ===")

  model.train()
  train_loss = 0

  for batch in train_loader:
    input_ids, attention_mask, labels = [x.to(device) for x in batch]

    optimizer.zero_grad()
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = outputs.logits

    loss = loss_func(logits, labels.float())
    loss.backward()

    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

    optimizer.step()
    scheduler.step()

    train_loss += loss.item()

  avg_train_loss = train_loss / len(train_loader)
  print(f"Training Loss: {avg_train_loss:.4f}")


model.eval()
val_loss = 0
predictions = []
targets = []

with torch.no_grad():
  for batch in val_loader:
    input_ids, attention_mask, labels = [x.to(device) for x in batch]

    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = outputs.logits

    loss = loss_func(logits, labels.float())
    val_loss += loss.item()

    pred = (torch.sigmoid(logits).cpu().numpy() > 0.5).astype(int)
    predictions.append(pred)
    targets.append(labels.cpu().numpy())

avg_val_loss = val_loss / len(val_loader)

# Convert correctly
predictions = np.vstack(predictions)
targets = np.vstack(targets)

f1 = f1_score(targets, predictions, average="micro")

print(f"Validation Loss: {avg_val_loss:.4f}")
print(f"Validation F1:   {f1:.4f}")

print("\n========== TESTING MODEL ==========\n")

model.eval()
test_loss = 0
predictions = []
targets = []

with torch.no_grad():
  for batch in test_loader:
    input_ids, attention_mask, labels = [x.to(device) for x in batch]

    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = outputs.logits

    loss = loss_func(logits, labels.float())
    test_loss += loss.item()

    pred = (torch.sigmoid(logits).cpu().numpy() > 0.5).astype(int)
    predictions.append(pred)
    targets.append(labels.cpu().numpy())

avg_test_loss = test_loss / len(test_loader)

predictions = np.vstack(predictions)
targets = np.vstack(targets)

f1 = f1_score(targets, predictions, average="micro")

print(f"Test Loss: {avg_test_loss:.4f}")
print(f"Test F1:   {f1:.4f}")

# **Text Generator**

In [ ]:
### Load and prepare data ###
df_text_gen = pd.read_csv("data/cuad_flattened_text_generation.csv")
df_text_gen["answer"] = df_text_gen["answer"].fillna("").str.strip()
df_text_gen = df_text_gen[df_text_gen["answer"].str.len() > 1]

# Input format
df_text_gen["input_text"] = df_text_gen.apply(
    lambda row: f"Extract {row['category']} from the following clause:\n{row['text']}",
    axis=1
)

print(f"Total examples: {len(df_text_gen)}\n")
print("\nCategory Distribution:")
print(df_text_gen["category"].value_counts())

### Split into train, validation, and test set ###
texts = df_text_gen["input_text"].tolist()

# 80% train, 10% validation, 10% test split
train_texts, val_test_texts = train_test_split(texts, test_size=0.2, random_state=0)
val_texts, test_texts = train_test_split(val_test_texts, test_size=0.5, random_state=0)

train_df = df_text_gen[df_text_gen["input_text"].isin(train_texts)].reset_index(drop=True)
val_df = df_text_gen[df_text_gen["input_text"].isin(val_texts)].reset_index(drop=True)
test_df = df_text_gen[df_text_gen["input_text"].isin(test_texts)].reset_index(drop=True)

print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))

### Tokenize inputs and targets ###
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")

def tokenize_texts(input_texts, target_texts, max_input_length=512, max_target_length=128):
    input_encodings = tokenizer(
        input_texts.to_list(),
        padding="max_length",
        truncation=True,
        max_length=max_input_length,
        return_tensors="pt"
    )
    target_encodings = tokenizer(
        target_texts.to_list(),
        padding="max_length",
        truncation=True,
        max_length=max_target_length,
        return_tensors="pt"
    )

    # Replace padding tokens with -100 to ignore in loss calcualtion
    labels = target_encodings["input_ids"].clone()
    labels[labels==tokenizer.pad_token_id] = -100

    return input_encodings, labels

# Convert texts and labels to tensors
train_input_encodings, train_labels = tokenize_texts(train_df["input_text"], train_df["answer"])
val_input_encodings, val_labels = tokenize_texts(val_df["input_text"], val_df["answer"])
test_input_encodings, test_labels = tokenize_texts(test_df["input_text"], test_df["answer"])

# Create datasets
train_dataset = TensorDataset(train_input_encodings["input_ids"], train_input_encodings["attention_mask"], train_labels)
val_dataset = TensorDataset(val_input_encodings["input_ids"], val_input_encodings["attention_mask"], val_labels)
test_dataset = TensorDataset(test_input_encodings["input_ids"], test_input_encodings["attention_mask"], test_labels)

# Dataloaders
batch_size = 8

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

### Load Model ###
model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-base")

# Add LoRA
# LoRA basically uses a specific set of weights such that
# since the amount of data is not a lot, updating all the model's
# weights can risk having it overfit to it, therefoer only updating
# a particular subset of it. This does not heed performance or actual output.
USE_LORA = True
if USE_LORA:
  lora_config = LoraConfig(
      r=8,
      lora_alpha=16,
      target_modules=["q", "v"],
      lora_dropout=0.05,
      bias="none",
      task_type="SEQ_2_SEQ_LM"
  )
  model = get_peft_model(model, lora_config)
  model.print_trainable_parameters()

model.config.use_cache = False
model = model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

# Scheduler
epochs = 3
total_steps = len(train_loader) * epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=100,
    num_training_steps=total_steps
)

### Training Loop ###
for epoch in range(epochs):
  print(f"\n=== Epoch {epoch+1}/{epochs} ===")

  model.train()
  train_loss = 0
  for batch in train_loader:
      input_ids, attention_mask, labels = [x.to(device) for x in batch]

      optimizer.zero_grad()
      outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)

      loss = outputs.loss
      loss.backward()
      torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

      optimizer.step()
      scheduler.step()

      train_loss += loss.item()

  avg_train_loss = train_loss / len(train_loader)
  print(f"Training Loss: {avg_train_loss:.4f}")

In [ ]:
### Validation Loop ###
model.eval()
val_loss = 0
predictions = []
targets = []

with torch.no_grad():
  for batch in val_loader:
    input_ids, attention_mask, labels = [x.to(device) for x in batch]

    outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
    val_loss += outputs.loss

    generated_ids = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_length=128,
        num_beams=4,
        early_stopping=True
    )

    # Decode predictions and labels
    pred = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
    labels_decoded = labels.clone()
    labels_decoded[labels_decoded == -100] = tokenizer.pad_token_id
    target = tokenizer.batch_decode(labels_decoded, skip_special_tokens=True)

    predictions.extend(pred)
    targets.extend(target)

avg_val_loss = val_loss / len(val_loader)
print(f"Validation Loss: {avg_val_loss:.4f}")

# ROUGE scores (validation)
rouge = evaluate.load("rouge")
rouge_scores = rouge.compute(predictions=predictions, references=targets)
print(f"Validation ROUGE-L: {rouge_scores['rougeL']:.4f}")

### Test Loop ###
model.eval()
test_loss = 0
predictions = []
targets = []
categories = []

with torch.no_grad():
  for i, batch in enumerate(test_loader):
    input_ids, attention_mask, labels = [x.to(device) for x in batch]

    outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
    test_loss += outputs.loss

    generated_ids = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_length=128,
        num_beams=4,
        early_stopping=True
    )

    # Decode predictions and labels
    pred = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
    labels_decoded = labels.clone()
    labels_decoded[labels_decoded == -100] = tokenizer.pad_token_id
    target = tokenizer.batch_decode(labels_decoded, skip_special_tokens=True)

    predictions.extend(pred)
    targets.extend(target)

    batch_size_real = len(pred)
    start = i * batch_size
    batch_categories = test_df.iloc[start:(start + batch_size_real)]["category"].tolist()
    categories.extend(batch_categories)

avg_test_loss = test_loss / len(test_loader)
print(f"Test Loss: {avg_test_loss:.4f}")

# ROUGE scores (overall)
rouge_scores = rouge.compute(predictions=predictions, references=targets)
print(f"Test ROUGE-1: {rouge_scores['rouge1']:.4f}")
print(f"Test ROUGE-2: {rouge_scores['rouge2']:.4f}")
print(f"Test ROUGE-L: {rouge_scores['rougeL']:.4f}")


# Perforamnce per-category
print("\n=== Performance per-category ===")
category_results = defaultdict(lambda: {"predictions": [], "targets": []})
for pred, target, cat in zip(predictions, targets, categories):
  category_results[cat]["predictions"].append(pred)
  category_results[cat]["targets"].append(target)

for category in sorted(category_results.keys()):
  cat_preds = category_results[category]["predictions"]
  cat_targets = category_results[category]["targets"]
  cat_rouge = rouge.compute(predictions=cat_preds, references=cat_targets)
  print(f"\n{category}:")
  print(f"  Samples: {len(cat_preds)}")
  print(f"  ROUGE-L: {cat_rouge['rougeL']:.4f}")

# Sample predictions
print("\n=== Sample Predictions ===")
for i in range(min(5, len(predictions))):
  print(f"\nExample {i+1} - {categories[i]}:")
  print(f"  Prediction: {predictions[i]}")
  print(f"  Actual:     {targets[i]}")

# **Model Push**

In [ ]:
# Push model to huggingface hub
from huggingface_hub import login

# Login (only once per session)
login()

model.push_to_hub("ClauseWise/legalbert-clause-classifier")
tokenizer.push_to_hub("ClauseWise/legalbert-clause-classifier")

In [ ]:
from huggingface_hub import login
login()
repo_id = "ClauseWise/flan-t5-cuad-clause-extractor-lora"

model.push_to_hub(repo_id)
tokenizer.push_to_hub(repo_id)

# **Smoke Test**

Verifies the pushed models load correctly from the Hub via `src/` and produce output.
Run after pushing to confirm inference works before moving on to deployment.

In [ ]:
import sys
sys.path.insert(0, "/content/ClauseWise")

from src.classifier import classify_clause
from src.extractor import extract_clause

TEST_CLAUSE = (
    "Either party may terminate this agreement at any time upon 30 days written notice "
    "to the other party, without cause and without liability."
)

print("=== Classifier ===")
labels = classify_clause(TEST_CLAUSE)
print(f"Input:  {TEST_CLAUSE}")
print(f"Labels: {labels}")

print("\n=== Extractor ===")
result = extract_clause("Termination", TEST_CLAUSE)
print(f"Extraction: {result}")